<p class="lead">A graph is the right data structure when the information is in the relations rather than in the rows: which compounds hit which targets, which proteins interact, which molecules resemble which. This unit builds the laboratory's own drug-target network from a frozen ChEMBL extraction — 602 compounds, 479 human proteins and 16,047 measured activities — and then does to it what should be done to every network before it is interpreted: measure it, compare it with a null model that preserves what is trivial about it, test a specific hypothesis with a permutation, and evaluate link prediction with the same protocol discipline the modelling units imposed on regression. Two selection biases are built into the data and are measured here rather than hidden. The recurring lesson is that network statistics are almost never interpretable on their own: a projection with 100,146 edges means nothing until it is thresholded, and a link-prediction AUC of 0.93 falls to 0.56 when the negatives are chosen honestly.</p>

<div class="box"><h4>Resumen en español</h4>
<p>Esta unidad representa relaciones biomédicas como grafos con <code>igraph</code> y las analiza con rigor estadístico. Se construye el grafo bipartito fármaco-diana a partir de una tabla medida de ChEMBL (602 compuestos, 479 proteínas humanas, 16.047 actividades; el 93 % de los compuestos se une a más de una diana), se describen sus distribuciones de grado y sus centralidades, y se proyecta sobre las dianas: la proyección sin filtrar tiene densidad 0,875, de modo que hace falta extraer un esqueleto por coeficiente de Jaccard antes de interpretarla. Las comunidades detectadas por modularidad voraz son casi exclusivamente quinasas, lo que refleja el origen del conjunto de datos. Todos los estadísticos se contrastan con un modelo nulo bipartito que preserva el grado (12.810 aristas del esqueleto observadas frente a 2.079 ± 122 aleatorizadas). La pregunta de selectividad ROCK1 frente a ROCK2 se responde con los 276 compuestos medidos en ambas dianas: diferencia media de -0,50 unidades logarítmicas, correlación 0,87 y solo un 15,6 % de compuestos con más de diez veces de diferencia. La predicción de enlaces alcanza 0,927 de AUC con negativos uniformes y 0,561 con negativos emparejados por grado. Finalmente se comparan la similitud química y el reparto de dianas, y se discuten dos sesgos explícitos: los compuestos se eligieron por ser ligandos de ROCK1, y el grado mide sobre todo cuánto se ha ensayado cada compuesto.</p></div>

<div class="box key"><h4>Learning objectives</h4>
<ul class="objectives">
<li>Represent biomedical relations as graphs in <code>igraph</code> and compute the basic descriptors: degree, components, clustering, shortest paths and the common centralities, knowing when each is defined.</li>
<li>Project a bipartite drug-target graph onto its two sides, threshold the projection into an interpretable backbone, and read the result as a polypharmacology or selectivity statement.</li>
<li>Detect communities and interpret them against external annotation, allowing for the resolution limit, for missing annotation and for the modularity of random graphs.</li>
<li>Assess any network statistic against a null model that preserves the degree sequence, and test a specific neighbourhood hypothesis with a permutation.</li>
<li>Evaluate link-prediction baselines with an honest negative set, measure the selection biases of the data, and decide when a network figure adds nothing.</li>
</ul></div>

<div class="box bridge"><h4>How to read this unit</h4>
<p>Read it linearly with the code running; every number in the prose is printed by a chunk of this document. In a fresh Google Colab R runtime run <code>install.packages(c("data.table","igraph","Matrix","patchwork"))</code> once before the first chunk. <b>Nothing is downloaded</b>: the graphs come from frozen tables, and the places where a real project would obtain a protein-protein or knowledge graph — STRING<sup class="c"><a href="#r7">[7]</a></sup>, Open Targets<sup class="c"><a href="#r20">[20]</a></sup>, Hetionet<sup class="c"><a href="#r6">[6]</a></sup> — are named but not queried. This is an <b>extended</b> unit for students working on target prioritisation, selectivity or systems-level questions. It uses the fingerprints of <a href="../U13_Features_Dimensionality/index.html">Unit 13</a>, the clustering of <a href="../U15_Clustering/index.html">Unit 15</a>, the evaluation protocol of <a href="../U12_Validation/index.html">Unit 12</a> and the permutation logic of <a href="../U8_Hypothesis_Testing/index.html">Unit 8</a>; it knits in about two minutes.</p></div>



In [ ]:
suppressPackageStartupMessages({
  library(data.table); library(igraph); library(Matrix); library(ggplot2); library(patchwork)
})
set.seed(0)
PAL <- c(blue = "#2b5d9b", orange = "#d2691e", green = "#3a7d44", purple = "#6a4c93")
theme_set(theme_minimal(base_size = 9) +
          theme(plot.title = element_text(face = "bold", size = 10), panel.grid.minor = element_blank()))
cat("R", paste(R.version$major, R.version$minor, sep = "."),
    "| igraph", as.character(packageVersion("igraph")),
    "| data.table", as.character(packageVersion("data.table")),
    "| Matrix", as.character(packageVersion("Matrix")),
    "| ggplot2", as.character(packageVersion("ggplot2")), "\n")



`igraph` replaces the Python edition's NetworkX; it is the standard graph library of
R<sup class="c"><a href="#r30">[30]</a></sup> (and of Python and C), the figures are drawn with
`ggplot2`<sup class="c"><a href="#r31">[31]</a></sup> and base graphics, its
algorithms are compiled, and the two libraries agree on every descriptor computed here except where noted. One
difference is worth stating in advance: `igraph::betweenness` computes the exact value on a graph of this size in a
fraction of a second, while the Python edition used a 120-pivot approximation, so the betweenness numbers below are
exact and differ from those of the Python page.

## 1. What the laboratory means by a network

<div class="box bridge"><h4>Domain context</h4>
<p>A <b>graph</b> is a set of <b>nodes</b> joined by <b>edges</b>. In drug discovery the nodes are usually compounds,
proteins, diseases or genes. A <b>drug-target network</b> joins a compound to every protein it is known to act
on<sup class="c"><a href="#r3">[3]</a></sup>; it is <b>bipartite</b>, meaning edges only ever run between the two
kinds of node. An edge here means a measured affinity: <b>pChEMBL</b> is -log<sub>10</sub> of the molar concentration
at which the compound acts (IC50, Ki or Kd), so 7 means 100 nM and each unit is a factor of ten.
<b>Polypharmacology</b> is the observation that most drugs bind several proteins<sup class="c"><a href="#r2">[2]</a></sup>,
sometimes as the mechanism of action and sometimes as the mechanism of a side effect; <b>hERG</b>, a cardiac
potassium channel, is the classic unwanted target. The opposite property, <b>selectivity</b>, is what a project wants
when the off-target is harmful — and the hardest case is a close <b>isoform</b>, two proteins from the same gene
family with nearly identical binding sites, such as <b>ROCK1</b> and <b>ROCK2</b>, the two Rho-associated kinases the
group works on. A <b>protein-protein interaction (PPI) network</b> joins proteins that interact physically or
functionally; <b>network medicine</b><sup class="c"><a href="#r1">[1]</a></sup> studies disease through
neighbourhoods of such a graph. A <b>chemical-similarity network</b> joins two molecules whose fingerprint similarity
exceeds a threshold, and a <b>Murcko scaffold</b><sup class="c"><a href="#r27">[27]</a></sup> is the ring framework of
a molecule, the usual label of a chemical series.</p></div>

The network analysed here is measured, not inferred. `drug_target_edges.csv` holds 16,047 activity records from
ChEMBL<sup class="c"><a href="#r28">[28]</a></sup>: for each of 602 well-characterised ROCK1 compounds, every human
single-protein target at which it has a pChEMBL value, with the median potency and the number of measurements behind
it. `drug_target_targets.csv` describes the 479 targets. Two properties of the extraction must be stated before any
interpretation, and both become exercises below.

<ol>
<li><b>It is a ROCK neighbourhood, not pharmacological space.</b> The compounds entered the table because they are
ROCK1 ligands, so the targets that appear are the ones ROCK inhibitors were tested against — overwhelmingly other
kinases. Nothing here licenses a statement about drugs in general.</li>
<li><b>Degree partly measures assay effort.</b> A compound linked to 300 targets has usually been through a kinase
selectivity panel; a compound linked to two has not. Absence of an edge is absence of a measurement, not evidence of
inactivity.</li>
</ol>



In [ ]:
REPO <- "DanielBustosG/DanielBustosG.github.io"   # GitHub repository hosting this course (set once in common/config.json)
DATA_URL <- paste0("https://raw.githubusercontent.com/", REPO, "/main/DataScience/data/")
data_file <- function(name) {   # local ../data/<name>; in Colab, downloaded once into data/
  local <- file.path("..", "data", name)
  if (file.exists(local)) return(local)
  dest <- file.path("data", name)
  if (!file.exists(dest)) {
    dir.create("data", showWarnings = FALSE)
    download.file(paste0(DATA_URL, name), dest, quiet = TRUE, mode = "wb")
  }
  dest
}
dp <- function(f) data_file(f)      # knitr runs with cwd = this unit's folder
edges <- fread(dp("drug_target_edges.csv"))
tg    <- fread(dp("drug_target_targets.csv"))
compounds <- sort(unique(edges$molecule_chembl_id)); targets <- sort(unique(edges$target_chembl_id))
g <- graph_from_data_frame(edges[, .(molecule_chembl_id, target_chembl_id, weight = pchembl_median)],
                           directed = FALSE)
V(g)$type <- V(g)$name %in% targets               # igraph marks the two sides with a logical 'type'
cdeg <- degree(g, compounds); tdeg <- degree(g, targets)
tname <- setNames(tg$target_name, tg$target_chembl_id)
cat(sprintf("edges table %d x %d | targets table %d x %d\n", nrow(edges), ncol(edges), nrow(tg), ncol(tg)))
cat(sprintf("bipartite graph: %d nodes (%d compounds + %d targets), %d edges, density %.4f, %d connected component(s)\n",
            vcount(g), length(compounds), length(targets), ecount(g),
            2 * ecount(g) / (vcount(g) * (vcount(g) - 1)), components(g)$no))
cat(sprintf("compound degree: median %d, mean %.1f, max %d | %d of %d compounds (%.0f %%) bind more than one target\n",
            median(cdeg), mean(cdeg), max(cdeg), sum(cdeg > 1), length(cdeg), 100 * mean(cdeg > 1)))
cat(sprintf("target degree  : median %d, mean %.1f, max %d | %d targets have fewer than 10 compounds\n",
            median(tdeg), mean(tdeg), max(tdeg), sum(tdeg < 10)))
knitr::kable(data.table(target = names(sort(tdeg, decreasing = TRUE))[1:6],
                        name = substr(tname[names(sort(tdeg, decreasing = TRUE))[1:6]], 1, 52),
                        compounds = sort(tdeg, decreasing = TRUE)[1:6]),
             caption = "Busiest targets")


In [ ]:
ROCK1 <- "CHEMBL3231"; ROCK2 <- "CHEMBL2973"
kin <- grepl("kinase", tname[targets], ignore.case = TRUE)
cat(sprintf("pChEMBL values: range %.2f to %.2f, median %.2f\n",
            min(edges$pchembl_median), max(edges$pchembl_median), median(edges$pchembl_median)))
cat(sprintf("edges that touch ROCK1 or ROCK2: %.1f %% of all edges\n",
            100 * (tdeg[[ROCK1]] + tdeg[[ROCK2]]) / ecount(g)))
cat(sprintf("targets whose NAME contains 'kinase': %d of %d (%.0f %%); they carry %.0f %% of all edges\n",
            sum(kin), length(targets), 100 * mean(kin), 100 * sum(tdeg[targets[kin]]) / ecount(g)))
cat(sprintf("ChEMBL protein_class annotation present for %d of %d targets\n",
            sum(!is.na(tg$protein_class) & tg$protein_class != ""), nrow(tg)))
nm_c <- edges[, .(n = sum(n_measurements), d = .N), by = molecule_chembl_id]
nm_t <- edges[, .(n = sum(n_measurements), d = .N), by = target_chembl_id]
cat(sprintf("\nassay-effort bias: Spearman(compound degree, total measurements) = %.3f\n",
            cor(nm_c$d, nm_c$n, method = "spearman")))
cat(sprintf("                   Spearman(target degree, total measurements)   = %.3f\n",
            cor(nm_t$d, nm_t$n, method = "spearman")))



The extraction is a single connected component of 1,081 nodes: every compound reaches every other through shared
targets. Ninety-three per cent of the compounds bind more than one target — genuine, measured polypharmacology. The
degree distributions are extremely skewed on both sides: the median compound has 5 targets and the busiest has 386,
the median target has 26 compounds and ROCK1 has 344.

The two biases are now numbers rather than warnings. Targets whose name contains "kinase" make up 79 % of the nodes
and carry 83 % of the edges, so this graph is a kinase graph; and the rank correlation between a compound's degree
and the total number of measurements behind its edges is **0.93** (0.94 on the target side), so degree is very largely
a record of how much work has been done. Note also that ChEMBL's `protein_class` column is empty for all 479 targets
in this export — a complete annotation gap, which section 4 has to work around with the target names themselves rather
than pretending the annotation exists.

<div class="exercise"><h4>Exercise 1</h4>
<p>Quantify the ROCK-neighbourhood bias. Compute, for every target, the fraction of its compounds that are also
measured at ROCK1, and plot that fraction against the target's degree. Then <code>delete_vertices()</code> the two
ROCK kinases and recompute <code>components()</code> and the median compound degree. How much of the graph's
connectivity is carried by the two targets the data set was built around, and what would you have to collect to turn
this into a sample of pharmacological space?</p></div>

## 2. Degrees, centralities and what they mean in a bipartite graph



In [ ]:
btw <- betweenness(g, normalized = TRUE)            # exact; igraph is fast enough at this size
eig <- eigen_centrality(g)$vector                   # igraph scales the vector so that the maximum is 1
top_b <- names(sort(btw, decreasing = TRUE))[1:6]
for (n in top_b)
  cat(sprintf("   %-9s %-14s degree %4d | betweenness %.4f | %s\n",
              ifelse(n %in% targets, "target", "compound"), n, degree(g, n), btw[[n]],
              substr(ifelse(n %in% targets, tname[[n]], ""), 1, 40)))
cat("\ntop eigenvector centrality among targets:\n")
for (n in names(sort(eig[targets], decreasing = TRUE))[1:4])
  cat(sprintf("   %-14s degree %4d | eigenvector %.3f | %s\n", n, degree(g, n), eig[[n]], substr(tname[[n]], 1, 46)))
cat(sprintf("\nSpearman(degree, betweenness) over all nodes: %.3f\n",
            cor(degree(g), btw, method = "spearman")))


In [ ]:
A  <- sparseMatrix(i = match(edges$molecule_chembl_id, compounds),
                   j = match(edges$target_chembl_id, targets), x = 1,
                   dims = c(length(compounds), length(targets)))
jacc <- function(M) {                               # Jaccard between all pairs of rows of a 0/1 matrix
  C <- as.matrix(tcrossprod(M)); d <- diag(C); U <- outer(d, d, "+") - C
  J <- ifelse(U > 0, C / U, 0); diag(J) <- 0; J
}
Jc <- jacc(A); Jt <- jacc(t(A))                     # compound-compound and target-target Jaccard
Cc <- as.matrix(tcrossprod(A)); Ct <- as.matrix(tcrossprod(t(A)))
cc_node <- function(J, C) sapply(seq_len(nrow(J)), function(i) {
  nb <- setdiff(which(C[i, ] > 0), i)               # the two-hop neighbourhood
  if (!length(nb)) NA_real_ else mean(J[i, nb])
})
cc_all <- c(cc_node(Jc, Cc), cc_node(Jt, Ct))
cat(sprintf("bipartite average clustering (dot mode, written out): %.3f over %d nodes\n",
            mean(cc_all, na.rm = TRUE), sum(!is.na(cc_all))))



<div class="box bridge"><h4>Substitution — the bipartite clustering coefficient by hand</h4>
<p>NetworkX ships <code>bipartite.average_clustering(B, mode="dot")</code>; <code>igraph</code> has no bipartite
clustering coefficient, because <code>transitivity()</code> counts triangles and a bipartite graph has none. The
definition is short enough to write out, which is what the chunk above does: for two nodes on the same side the "dot"
overlap is the Jaccard coefficient of their neighbourhoods, \(|N(u)\cap N(v)|/|N(u)\cup N(v)|\), the clustering of a
node is the mean overlap with every node two hops away, and the graph's coefficient is the mean over all
nodes<sup class="c"><a href="#r33">[33]</a></sup>. The value, 0.223, matches the Python edition to three decimals —
and the matrices <code>Jc</code> and <code>Jt</code> it builds on the way are the projections used in section 3, so
nothing is wasted.</p></div>



In [ ]:
ddf <- rbind(data.table(side = "compounds", k = as.integer(names(table(cdeg))), n = as.integer(table(cdeg))),
             data.table(side = "targets",   k = as.integer(names(table(tdeg))), n = as.integer(table(tdeg))))
pd1 <- ggplot(ddf, aes(k, n, colour = side)) + geom_point(size = 0.9, alpha = 0.7) +
  scale_x_log10() + scale_y_log10() +
  scale_colour_manual(values = c(compounds = PAL[["blue"]], targets = PAL[["orange"]]), name = NULL) +
  labs(title = "Degree distributions, both sides", x = "degree k", y = "number of nodes") +
  theme(legend.position = c(0.8, 0.85), legend.text = element_text(size = 6))
pd2 <- ggplot(nm_c, aes(d, n)) + geom_point(size = 0.7, alpha = 0.45, colour = PAL[["blue"]]) +
  scale_x_log10() + scale_y_log10() +
  labs(title = "Degree is assay effort (rho = 0.93)",
       x = "compound degree (targets)", y = "total measurements behind its edges")
pd3 <- ggplot(edges, aes(pchembl_median)) + geom_histogram(bins = 40, fill = PAL[["blue"]]) +
  geom_vline(xintercept = median(edges$pchembl_median), linetype = "dotted") +
  labs(title = "Edge weights: measured potency", x = "pChEMBL of an edge", y = "edges")
print(pd1 | pd2 | pd3)



<p class="caption"><b>Figure 1.</b> Left: both degree distributions are heavy-tailed on logarithmic axes — a few
compounds have been profiled against hundreds of targets and a few targets have been screened against hundreds of
compounds — but neither is a clean power law<sup class="c"><a href="#r15">[15]</a></sup>, and with 602 and 479 nodes no
such claim could be supported anyway. Centre: compound degree against the number of measurements behind its edges,
Spearman 0.93; the "most promiscuous" compounds are first of all the most assayed ones. Right: the pChEMBL
distribution of the edges, median 6.10, i.e. the typical recorded interaction is micromolar-to-submicromolar.</p>

Centrality in a bipartite graph needs care. Exact betweenness is dominated by ROCK1 (0.314), with ROCK2 second
(0.096) and the four most-profiled compounds behind them — exactly what a graph assembled around two targets must look
like, and a reminder that a centrality ranking reproduces the sampling design before it reveals biology. (The Python
edition's pivot approximation put ROCK1 and ROCK2 almost level at 0.220 and 0.210; the exact computation shows that
ROCK1 alone carries three times as many shortest paths, which is a better description of the same graph and a small
lesson about approximate centralities.) Eigenvector centrality, which is defined here because the graph is connected,
puts the tyrosine kinases FLT3 (0.447 on igraph's max-one scale), RET (0.411), LCK (0.391) and ABL1 (0.381) at the
top: they are the targets most often co-measured with the hubs. The rank correlation between degree and betweenness
over all nodes is 0.86, and the bipartite clustering coefficient is 0.223.

## 3. Projecting onto the targets, and why a projection must be thresholded

The bipartite graph answers questions about compounds and targets together. To ask *which targets share chemistry*,
project it: join two targets when at least one compound binds both. The weight can be the number of shared compounds
or — better, because it is scale-free — the Jaccard coefficient of their compound sets,
\(|C_i \cap C_j| / |C_i \cup C_j|\), which the matrices of the previous chunk already contain.

The projection of a dense bipartite graph is nearly complete, and this one is no exception, so the first operation
after projecting is always to threshold. Keeping only edges above a Jaccard cut-off produces a
**backbone**<sup class="c"><a href="#r32">[32]</a></sup>; the cut is a free parameter and must be reported with the
result, exactly like the similarity threshold of a chemical network.



In [ ]:
iu <- upper.tri(Jt); jw <- Jt[iu]; sw <- Ct[iu]
cat(sprintf("unthresholded target projection: %d nodes, %d edges, density %.3f\n",
            length(targets), sum(jw > 0), sum(jw > 0) / choose(length(targets), 2)))
cat(sprintf("shared-compound weights (pairs sharing at least one): median %d, 90th percentile %d, maximum %d\n",
            median(sw[sw > 0]), quantile(sw[sw > 0], 0.9), max(sw)))
cat(sprintf("Jaccard weights: median %.3f, 90th percentile %.3f, maximum %.3f | %d pairs share no compound\n",
            median(jw[jw > 0]), quantile(jw[jw > 0], 0.9), max(jw), sum(jw == 0)))
thr_tab <- rbindlist(lapply(c(0.2, 0.3, 0.4, 0.5), function(thr) {
  Gh <- graph_from_adjacency_matrix(Jt >= thr, mode = "undirected", diag = FALSE)
  data.table(threshold = thr, edges = ecount(Gh),
             density = ecount(Gh) / choose(length(targets), 2),
             isolated = sum(degree(Gh) == 0), components = components(Gh)$no)
}))
knitr::kable(thr_tab, digits = 3)
i1 <- match(ROCK1, targets); i2 <- match(ROCK2, targets)
cat(sprintf("\nROCK1 and ROCK2 share %d compounds (Jaccard %.3f)\n", Ct[i1, i2], Jt[i1, i2]))
strong <- which(Ct * upper.tri(Ct) > 0, arr.ind = TRUE)
strong <- strong[order(-Ct[strong])[1:5], ]
for (k in 1:5)
  cat(sprintf("   %-32s -- %-32s %3d shared\n", substr(tname[targets[strong[k, 1]]], 1, 32),
              substr(tname[targets[strong[k, 2]]], 1, 32), Ct[strong[k, 1], strong[k, 2]]))


In [ ]:
ths <- seq(0.05, 0.8, length.out = 40)
pj1 <- ggplot(data.table(w = jw[jw > 0]), aes(w)) + geom_histogram(bins = 60, fill = PAL[["blue"]]) +
  geom_vline(xintercept = 0.3, linetype = "dotted", colour = PAL[["orange"]]) +
  annotate("text", x = 0.33, y = 6000, label = "backbone cut", size = 2.4, colour = PAL[["orange"]], hjust = 0) +
  labs(title = "Most pairs share a little, a few share a lot",
       x = "Jaccard weight of a target pair", y = "pairs")
pj2 <- ggplot(data.table(t = ths, n = sapply(ths, function(x) sum(jw >= x))), aes(t, n)) +
  geom_line(colour = PAL[["blue"]]) + scale_y_log10() +
  geom_vline(xintercept = 0.3, linetype = "dotted", colour = PAL[["orange"]]) +
  labs(title = "Choosing the backbone", x = "Jaccard threshold", y = "edges kept (log)")
print(pj1 | pj2)



<p class="caption"><b>Figure 2.</b> Left: the distribution of Jaccard weights over the 100,146 target pairs that share
at least one compound (of the 114,481 pairs 479 targets admit; the remaining 14,335 share none and carry weight zero) —
median 0.154, ninetieth percentile 0.318, maximum 1.000 — showing that the projection is dense in weak links and
sparse in strong ones. Right: the number of edges surviving each threshold, on a logarithmic scale; the cut at 0.3
used below is marked. There is no natural break in either plot, which is why the threshold is a declared choice and
not a discovery.</p>

Unthresholded, the projection has 100,146 edges among 479 targets — a density of 0.875, which is to say almost every
pair of targets shares at least one compound and the graph carries no usable structure. At Jaccard 0.3 the backbone
keeps 12,810 edges (density 0.112) and leaves 42 targets isolated; at 0.5 only 874 edges survive and 171 targets drop
out. We work at 0.3 and say so. The strongest single link is ROCK1 - ROCK2 with 276 shared compounds and a Jaccard of
0.730, which is the pair the next-but-one section is about.

## 4. Communities and the annotation gap

Community detection partitions a graph into groups denser inside than between. Greedy modularity
maximisation<sup class="c"><a href="#r14">[14]</a></sup> — `igraph::cluster_fast_greedy`, the Clauset-Newman-Moore
algorithm — optimises \(Q\)<sup class="c"><a href="#r12">[12]</a></sup>, the fraction of edges inside communities
minus the fraction expected in a degree-preserving random graph; `cluster_louvain` and the Leiden
refinement<sup class="c"><a href="#r4">[4]</a></sup><sup class="c"><a href="#r5">[5]</a></sup> optimise the same
objective faster on larger graphs. Interpreting the communities requires external labels, and here the obvious label
is missing: ChEMBL's protein classification came back empty for every target in this export. The honest workaround is
to use the target names as a crude, text-matched surrogate and to say that is what we are doing.



In [ ]:
Gb <- graph_from_adjacency_matrix(Jt >= 0.3, mode = "undirected", diag = FALSE)
V(Gb)$name <- targets
core <- induced_subgraph(Gb, which(degree(Gb) > 0))
cm   <- cluster_fast_greedy(core)
cl   <- communities(cm)[order(-lengths(communities(cm)))]
cat(sprintf("backbone at Jaccard >= 0.3: %d targets, %d edges | %d communities, Q = %.3f\n",
            vcount(core), ecount(core), length(cl), modularity(cm)))
cat(sprintf("Louvain on the same backbone: %d communities, Q = %.3f\n",
            length(cluster_louvain(core)), modularity(cluster_louvain(core))))
comm_tab <- rbindlist(lapply(seq_along(cl), function(i) {
  nms <- tname[cl[[i]]]
  data.table(community = i, size = length(cl[[i]]),
             kinase_named = sprintf("%.0f %%", 100 * mean(grepl("kinase", nms, ignore.case = TRUE))),
             example = paste(substr(nms[1:min(2, length(nms))], 1, 32), collapse = "; "))
}))
knitr::kable(head(comm_tab, 8))
nonkin <- cl[sapply(cl, function(c_) !any(grepl("kinase", tname[c_], ignore.case = TRUE)))]
cat(sprintf("communities with no kinase-named member: %d (sizes %s)\n",
            length(nonkin), paste(lengths(nonkin), collapse = ", ")))
cat("largest of them:", paste(substr(tname[nonkin[[1]]], 1, 34), collapse = " | "), "\n")


In [ ]:
ord <- unlist(cl)
Adj <- as.matrix(as_adjacency_matrix(core))[ord, ord]
par(mfrow = c(1, 2), mar = c(4, 4, 2.4, 1), cex = 0.7)
image(seq_len(nrow(Adj)), seq_len(nrow(Adj)), t(Adj[nrow(Adj):1, ]), col = c("white", "grey25"),
      xlab = "", ylab = "", axes = FALSE, main = "Backbone adjacency, ordered by community")
abline(v = cumsum(lengths(cl))[-length(cl)], col = PAL[["orange"]], lwd = 0.6)
abline(h = nrow(Adj) - cumsum(lengths(cl))[-length(cl)], col = PAL[["orange"]], lwd = 0.6)
box()
kf <- sapply(cl, function(c_) mean(grepl("kinase", tname[c_], ignore.case = TRUE)))
plot(lengths(cl), kf, log = "x", pch = 19, col = PAL[["blue"]], cex = 0.6 + 0.9 * sqrt(lengths(cl) / 180),
     xlab = "community size (log)", ylab = "fraction of members named 'kinase'", ylim = c(-0.05, 1.08),
     main = "Communities are kinase panels")



<p class="caption"><b>Figure 3.</b> Left: the adjacency matrix of the 437-target backbone with rows and columns
ordered by community; the dense diagonal blocks are the communities and the off-diagonal density shows how much they
still share. Right: each community's size against the fraction of its members whose name contains "kinase" — the four
largest communities are 79-100 % kinases, while six small communities (sizes 8, 4, 2, 2, 2 and 2) contain no
kinase-named target at all; the largest of those six holds the carbonic anhydrases.</p>

Seventeen communities at Q = 0.304, and the two largest hold 179 and 177 targets of which 91 % and 79 % are named
"kinase". A community of eight contains no kinase at all — six of its members are carbonic anhydrases — and a
four-member community holds cytochrome P450 isoforms: the counter-screens and ADME assays that accompany a kinase
programme. Louvain on the same backbone returns 13 communities at a slightly higher Q = 0.325, which is the expected
behaviour of a better optimiser on the same objective and a reminder that "the communities" of a graph are an
algorithm's output, not a property of the data. This is a correct description of the data and a poor description of
pharmacology — the communities recover the structure of a kinase selectivity panel, because that is what the compounds
were tested in. With a real protein-class annotation the same partition could be scored against a reference labelling
(an adjusted Rand index, as in Unit 15); with the annotation absent, a text match on the target name and an explicit
statement of its crudeness is as far as the data allow.

<div class="exercise"><h4>Exercise 2</h4>
<p>Rebuild the backbone at Jaccard 0.2 and 0.4, rerun <code>cluster_fast_greedy</code> and tabulate the number of
communities, the modularity and the kinase fraction of the largest community. Then check the resolution
limit<sup class="c"><a href="#r13">[13]</a></sup>: how small is the smallest community the algorithm returns at each
threshold, and could a four-target family be detected at all in a backbone of this size?</p></div>

## 5. Null models for a bipartite graph

Every number above — the density of the backbone, the modularity, the Jaccard of ROCK1 with ROCK2 — is shaped by the
degree sequence before it is shaped by biology. A target measured against 344 compounds will share compounds with
everything. The null model that removes that effect while keeping the degrees is **degree-preserving randomisation**:
repeatedly pick two edges and swap their endpoints<sup class="c"><a href="#r16">[16]</a></sup>. In a bipartite graph
the swap must exchange *target* endpoints between two compound-edges, otherwise it would create compound-compound
edges; the twelve-line implementation below does exactly that, and is the reason to write the null yourself rather
than call `rewire()` with a generic method.



In [ ]:
projection_stats <- function(M, thr = 0.3) {       # mean Jaccard over target pairs, and backbone size
  C <- as.matrix(tcrossprod(t(M))); d <- diag(C); U <- outer(d, d, "+") - C
  J <- ifelse(U > 0, C / U, 0); diag(J) <- 0; up <- upper.tri(J)
  c(mean_jaccard = mean(J[up]), backbone_edges = sum(J[up] >= thr))
}
bipartite_swap <- function(M, n_swaps) {
  ed <- which(M > 0, arr.ind = TRUE); Md <- as.matrix(M)
  for (s in seq_len(n_swaps)) {
    ij <- sample(nrow(ed), 2)
    c1 <- ed[ij[1], 1]; t1 <- ed[ij[1], 2]; c2 <- ed[ij[2], 1]; t2 <- ed[ij[2], 2]
    if (c1 == c2 || t1 == t2 || Md[c1, t2] || Md[c2, t1]) next
    Md[c1, t1] <- 0; Md[c2, t2] <- 0; Md[c1, t2] <- 1; Md[c2, t1] <- 1
    ed[ij[1], 2] <- t2; ed[ij[2], 2] <- t1
  }
  Md
}
observed <- projection_stats(A)
set.seed(0)
null <- t(sapply(1:20, function(i) projection_stats(bipartite_swap(A, 5 * sum(A)))))
cat(sprintf("mean Jaccard over target pairs : observed %.4f | null %.4f +/- %.4f | z = %.1f\n",
            observed[1], mean(null[, 1]), sd(null[, 1]), (observed[1] - mean(null[, 1])) / sd(null[, 1])))
cat(sprintf("backbone edges (Jaccard >= 0.3): observed %d | null %.0f +/- %.0f | z = %.1f\n",
            observed[2], mean(null[, 2]), sd(null[, 2]), (observed[2] - mean(null[, 2])) / sd(null[, 2])))


In [ ]:
pn1 <- ggplot(data.table(v = null[, 1]), aes(v)) + geom_histogram(bins = 10, fill = "grey70") +
  geom_vline(xintercept = observed[1], colour = PAL[["orange"]], linewidth = 1) +
  labs(title = "The average is almost fixed by the degrees",
       x = "mean Jaccard over all target pairs", y = "randomisations")
pn2 <- ggplot(data.table(v = null[, 2]), aes(v)) + geom_histogram(bins = 10, fill = "grey70") +
  geom_vline(xintercept = observed[2], colour = PAL[["orange"]], linewidth = 1) +
  coord_cartesian(xlim = c(0, 13500)) +
  labs(title = "The upper tail is not",
       x = "backbone edges at Jaccard >= 0.3 (orange: observed)", y = "randomisations")
print(pn1 | pn2)



<p class="caption"><b>Figure 4.</b> Observed statistics (orange line) against twenty degree-preserving randomisations
of the same bipartite graph (grey). Left: the mean Jaccard over all 114,481 target pairs is 0.1482 observed against
0.1424 +/- 0.0003 randomised — a z of 17, but an effect of 4 % that no one should build an argument on. Right: the
number of backbone edges at Jaccard 0.3 is 12,810 observed against 2,079 +/- 122 randomised, z = 88 and a six-fold
excess. The structure of this network lives entirely in the upper tail of the Jaccard distribution; the average is
essentially a restatement of the degree sequence.</p>

That contrast is the practical lesson of null modelling. Two summaries of the same projection, computed from the same
randomisations, support completely different conclusions: one says the targets share chemistry about as much as their
degrees force them to, the other says that specific pairs share far more than that. Report the statistic you tested,
the null that produced it and the number of randomisations — and prefer a statistic with a tail, because that is where
structure hides.

## 6. ROCK1 against ROCK2: a selectivity question asked of the data

Isoform selectivity is the question the group's ROCK programme actually faces. The bipartite graph says how many
compounds have been measured at both kinases; the edge weights say how potent each is at each. The selectivity of a
compound is the difference of the two pChEMBL values, and a difference of 1 means a tenfold preference.



In [ ]:
P  <- dcast(edges, molecule_chembl_id ~ target_chembl_id, value.var = "pchembl_median")
Pm <- as.matrix(P[, -1]); rownames(Pm) <- P$molecule_chembl_id      # compounds x targets, NA = not measured
ok <- !is.na(Pm[, ROCK1]) & !is.na(Pm[, ROCK2])
d  <- Pm[ok, ROCK1] - Pm[ok, ROCK2]
cat(sprintf("compounds measured at BOTH ROCK1 and ROCK2: %d (ROCK1 only: %d, ROCK2 only: %d)\n",
            sum(ok), sum(!is.na(Pm[, ROCK1]) & is.na(Pm[, ROCK2])),
            sum(is.na(Pm[, ROCK1]) & !is.na(Pm[, ROCK2]))))
cat(sprintf("pROCK1 - pROCK2: mean %+.3f, median %+.3f, sd %.3f\n", mean(d), median(d), sd(d)))
cat(sprintf("Pearson correlation of the two potencies r = %.3f (Spearman %.3f)\n",
            cor(Pm[ok, ROCK1], Pm[ok, ROCK2]), cor(Pm[ok, ROCK1], Pm[ok, ROCK2], method = "spearman")))
cat(sprintf("selective by more than 10-fold (|difference| > 1): %d compounds (%.1f %%)\n",
            sum(abs(d) > 1), 100 * mean(abs(d) > 1)))
cat(sprintf("selective by more than 3-fold  (|difference| > 0.5): %d compounds (%.1f %%)\n",
            sum(abs(d) > 0.5), 100 * mean(abs(d) > 0.5)))
cat(sprintf("most ROCK2-preferring: %s (%.2f log units) | most ROCK1-preferring: %s (%+.2f)\n",
            names(which.min(d)), min(d), names(which.max(d)), max(d)))


In [ ]:
sdt <- data.table(r1 = Pm[ok, ROCK1], r2 = Pm[ok, ROCK2], d = d)
lims <- range(c(sdt$r1, sdt$r2)) + c(-0.2, 0.2)
ps1 <- ggplot(sdt, aes(r2, r1)) + geom_point(size = 0.8, alpha = 0.6, colour = PAL[["blue"]]) +
  geom_abline(linetype = "dashed") +
  geom_abline(intercept = c(1, -1), linetype = "dotted", colour = PAL[["orange"]]) +
  coord_cartesian(xlim = lims, ylim = lims) +
  labs(title = "276 compounds measured at both", x = "pChEMBL at ROCK2", y = "pChEMBL at ROCK1")
ps2 <- ggplot(sdt, aes(d)) + geom_histogram(bins = 40, fill = PAL[["blue"]]) +
  geom_vline(xintercept = 0) +
  geom_vline(xintercept = c(-1, 1), linetype = "dotted", colour = PAL[["orange"]]) +
  labs(title = sprintf("Selectivity window: mean %+.2f", mean(d)),
       x = "pROCK1 - pROCK2 (log units)", y = "compounds")
nt  <- rowSums(!is.na(Pm)); broad <- names(nt)[nt >= 20]
gap <- sapply(broad, function(c_) {
  v <- sort(Pm[c_, ][!is.na(Pm[c_, ])], decreasing = TRUE); unname(v[1] - v[10])
})
ps3 <- ggplot(data.table(g = gap), aes(g)) + geom_histogram(bins = 30, fill = PAL[["green"]]) +
  geom_vline(xintercept = 0.5, linetype = "dotted") +
  labs(title = "Panel-wide selectivity index", x = "best minus 10th-best pChEMBL", y = "compounds")
print(ps1 | ps2 | ps3)



<p class="caption"><b>Figure 5.</b> Left: potency at ROCK1 against potency at ROCK2 for the 276 compounds measured at
both; the cloud sits on the diagonal (r = 0.870) and mostly inside the tenfold band (dotted). Centre: the distribution
of the difference, centred at -0.50 log units — this chemical series is on average slightly more potent at ROCK2 —
with 15.6 % of compounds beyond tenfold selectivity in one direction or the other and a single outlier at -3.55.
Right: the panel-wide index of the next paragraph, the gap between a compound's best and tenth-best potency, for the
174 compounds measured at twenty targets or more; only four fall below the 0.5 log units that would make a profile
genuinely flat.</p>

Three conclusions follow, and the third is the one a project needs. First, isoform selectivity within the ROCK family
is hard: the two kinases rank compounds almost identically (Spearman 0.851), as their near-identical ATP pockets
predict. Second, the few genuinely selective compounds are the interesting ones, and they are identifiable by
identifier from this table — which is how a selectivity programme starts. Third, the absence of an edge is not
evidence: 68 compounds are measured at ROCK1 and not at ROCK2 and 34 the other way round, and nothing in the graph
says whether those are inactive or simply untested. A selectivity claim needs the measurement, not the missing edge.



In [ ]:
cat(sprintf("compounds measured at 20 or more targets: %d\n", length(broad)))
cat(sprintf("selectivity index (best minus 10th best pChEMBL): median %.2f, 10th percentile %.2f, 90th percentile %.2f\n",
            median(gap), quantile(gap, 0.1), quantile(gap, 0.9)))
cat(sprintf("flat profiles (index < 0.5 log units, i.e. genuinely promiscuous): %d of %d (%.0f %%)\n",
            sum(gap < 0.5), length(gap), 100 * mean(gap < 0.5)))
cat(sprintf("Spearman(selectivity index, number of targets measured) = %.3f\n",
            cor(gap, nt[broad], method = "spearman")))
cat(sprintf("most selective broadly profiled compound: %s (index %.2f)\n", names(which.max(gap)), max(gap)))



A single pair of targets is not the whole selectivity question, and the same table answers the panel-wide version. For
the 174 compounds measured at twenty targets or more, the gap between the best and the tenth-best pChEMBL is a usable
selectivity index: its median is 1.47 log units, its tenth percentile 0.72 and its ninetieth 2.48, and only four
compounds (2 %) have a profile flat enough to call genuinely promiscuous. The index falls as the number of measured
targets grows (Spearman -0.379), which is the assay-effort bias again: the more panels a compound has been through,
the more likely one of them found a second potent target. Report a selectivity index together with the number of
targets it was computed over, exactly as an enrichment factor is reported with its prevalence.

## 7. Link prediction, and the negative set that decides the answer

Link prediction asks which absent edges are most likely to be present but unobserved — in a drug-target graph, which
compound-target pairs are untested but
plausible<sup class="c"><a href="#r18">[18]</a></sup><sup class="c"><a href="#r19">[19]</a></sup>. The protocol is that
of Unit 12 transplanted to graphs: hide a random 10 % of the edges, score candidate pairs using only the remaining
graph, and measure how well the hidden edges are recovered.

In a bipartite graph the familiar scores need one adjustment: a compound and a target have no common neighbours, so
the shortest informative path has length three (compound - target - compound - target). The three baselines below are
the natural bipartite analogues<sup class="c"><a href="#r26">[26]</a></sup>. **Path count** counts the three-step
paths between the pair, \((A A^{\mathsf T} A)_{ct}\) in matrix form. **Profile similarity** replaces the inner count
by the row-normalised profile overlap of the two compounds, so that a compound tested at everything does not dominate.
**Preferential attachment**<sup class="c"><a href="#r17">[17]</a></sup> multiplies the two degrees and ignores
structure entirely — it is in the panel as a control, because if it scores as well as the others, the others are
measuring popularity.

The negatives decide the answer. Uniformly drawn non-edges are mostly pairs of a rarely tested compound with a rarely
tested target, which any degree-aware score separates trivially. The honest alternative pairs each hidden edge's
compound with a target of **similar degree** to the true one, so that the popularity shortcut is removed.



In [ ]:
roc_auc <- function(y, s) {                      # Mann-Whitney form, as in Unit 19
  r <- rank(s); n1 <- sum(y); (sum(r[y == 1]) - n1 * (n1 + 1) / 2) / (n1 * (length(y) - n1))
}
Ad <- as.matrix(A)
set.seed(1)
present <- which(Ad > 0, arr.ind = TRUE)
pos <- present[sample(nrow(present), round(0.1 * nrow(present))), ]
Atr <- Ad; Atr[pos] <- 0                          # hide 10 % of the edges
An  <- Atr / pmax(rowSums(Atr), 1)
S_path <- Atr %*% (t(Atr) %*% Atr)                # three-step path count
S_prof <- (An %*% t(An)) %*% Atr                  # profile-similarity weighted
S_pref <- outer(rowSums(Atr), colSums(Atr))       # preferential attachment (degree product)
absent <- which(Ad == 0, arr.ind = TRUE)
easy_neg <- absent[sample(nrow(absent), 20 * nrow(pos)), ]
tdeg_tr <- colSums(Atr); ordt <- order(tdeg_tr)
set.seed(2)
pick <- pos[sample(nrow(pos), 20 * nrow(pos), replace = TRUE), ]
hard_neg <- do.call(rbind, lapply(seq_len(nrow(pick)), function(k) {
  c_ <- pick[k, 1]; t_ <- pick[k, 2]
  lo <- max(1, findInterval(tdeg_tr[t_], tdeg_tr[ordt]) - 15)
  cand <- ordt[lo:min(lo + 29, length(ordt))]; cand <- cand[Ad[c_, cand] == 0]
  if (!length(cand)) NULL else c(c_, cand[sample(length(cand), 1)])
}))
cat(sprintf("hidden edges %d | uniform negatives %d | degree-matched negatives %d\n",
            nrow(pos), nrow(easy_neg), nrow(hard_neg)))
evaluate <- function(neg, label) {
  cand <- rbind(pos, neg); y <- c(rep(1, nrow(pos)), rep(0, nrow(neg)))
  cat(sprintf("\n%s (prevalence %.3f)\n", label, mean(y)))
  out <- list()
  for (nm in c("3-step path count", "profile similarity", "preferential attachment")) {
    S <- switch(nm, `3-step path count` = S_path, `profile similarity` = S_prof, S_pref)
    s <- S[cand]
    out[[nm]] <- list(auc = roc_auc(y, s), y = y, s = s)
    cat(sprintf("   %-24s ROC-AUC %.3f | precision@%d %.3f\n", nm, out[[nm]]$auc, nrow(pos),
                mean(y[order(-s)][1:nrow(pos)])))
  }
  out
}
easy <- evaluate(easy_neg, "negatives drawn uniformly at random")
hard <- evaluate(hard_neg, "negatives matched on target degree")


In [ ]:
roc_pts <- function(y, s) {
  ys <- y[order(-s)]
  data.table(fpr = c(0, cumsum(1 - ys) / sum(1 - ys)), tpr = c(0, cumsum(ys) / sum(ys)))
}
mk <- function(res, title) {
  d <- rbindlist(lapply(names(res), function(nm)
    data.table(method = sprintf("%s (%.3f)", nm, res[[nm]]$auc), roc_pts(res[[nm]]$y, res[[nm]]$s))))
  ggplot(d, aes(fpr, tpr, colour = method)) + geom_line(linewidth = 0.6) +
    geom_abline(linetype = "dotted") +
    scale_colour_manual(values = unname(PAL[c("orange", "blue", "green")]), name = NULL) +
    labs(title = title, x = "false-positive rate", y = "true-positive rate") +
    theme(legend.position = c(0.62, 0.2), legend.text = element_text(size = 5.6))
}
print(mk(easy, "Link prediction, uniform negatives") | mk(hard, "Link prediction, degree-matched negatives"))



<p class="caption"><b>Figure 6.</b> The same three predictors, the same 1,605 hidden edges, two negative sets. Against
uniformly drawn non-edges the path count reaches ROC-AUC 0.927 and recovers 51 % of the top 1,605 pairs — but
preferential attachment, which sees nothing except the two degrees, reaches 0.921, which is the tell. Against
negatives matched on target degree, the path count falls to 0.561, profile similarity to 0.586 and preferential
attachment to 0.506: with the popularity shortcut removed, these baselines are barely better than guessing.</p>

The conclusion generalises far beyond graphs: **a benchmark is as hard as its negatives**. A published AUC of 0.93
for drug-target link prediction may be measuring nothing but which targets are frequently assayed; the only way to
find out is to re-run it against degree-matched negatives, which costs ten lines. Unit 19 made the same point with
property-matched decoys — here the matching variable is degree instead of molecular weight. The practical
recommendation for this data set is the one the numbers support: use the network to *rank which untested pair to
measure next* only if a degree-matched evaluation shows the ranking beats popularity, and otherwise admit that the
model is recommending the targets everyone already screens.

## 8. Does sharing targets mean looking alike?

The unit's second graph is the chemical-similarity network of Unit 15: compounds joined when their Morgan
fingerprints<sup class="c"><a href="#r29">[29]</a></sup> are similar. It earns its place here by answering a question
the drug-target graph cannot: is the target profile of a compound predictable from its structure? Similarity-based
target inference — the similarity ensemble approach<sup class="c"><a href="#r8">[8]</a></sup><sup class="c"><a href="#r9">[9]</a></sup>
and the pharmacological-space mapping that preceded it<sup class="c"><a href="#r10">[10]</a></sup> — assumes it is.
With measured profiles for 602 compounds we can check the assumption instead of inheriting it.

<div class="box bridge"><h4>Substitution — fingerprints from the frozen file</h4>
<p>R has no RDKit binding in this environment, so the Morgan fingerprints (radius 2, 2,048 bits) are read from
<code>data/rock1_morgan_sparse.csv.gz</code> — two columns, InChIKey and the index of a set bit — and rebuilt with
<code>Matrix::sparseMatrix()</code>, exactly as in <a href="../U19_Virtual_Screening/index.html">Unit 19</a>;
<a href="../../data/manifest_fingerprints.json">data/manifest_fingerprints.json</a> records the RDKit call that produced
them. The join needs care: the curated table keys structures by standardised parent InChIKey and one key can carry
several ChEMBL identifiers, so the 602 network compounds map onto 593 distinct keys and the mapping must be expanded
rather than matched one-to-one — a <code>merge</code> on the key, not a <code>match</code>.</p></div>



In [ ]:
cur <- fread(dp("rock1_chembl_curated.csv"))
sp  <- fread(dp("rock1_morgan_sparse.csv.gz"))
map <- rbindlist(lapply(seq_len(nrow(cur)), function(i)
  data.table(chembl_id = strsplit(gsub(" ", "", as.character(cur$chembl_ids[i])), "[|;]")[[1]],
             inchikey = cur$inchikey[i])))
ik <- setNames(map$inchikey, map$chembl_id)
cmp_have <- compounds[compounds %in% names(ik)]
cdt <- data.table(ci = seq_along(cmp_have), id = unname(ik[cmp_have]))
jn  <- merge(cdt, sp, by = "id", allow.cartesian = TRUE)
FP  <- sparseMatrix(i = jn$ci, j = jn$bit + 1L, x = 1, dims = c(length(cmp_have), 2048))
Tm  <- as.matrix(tcrossprod(FP)); a <- rowSums(FP)
Tn  <- Tm / (outer(a, a, "+") - Tm); diag(Tn) <- 0
Jsub <- Jc[match(cmp_have, compounds), match(cmp_have, compounds)]
up <- upper.tri(Tn); tv <- Tn[up]; jv <- Jsub[up]
cat(sprintf("curated structures available for %d of the %d network compounds (%d distinct InChIKeys)\n",
            length(cmp_have), length(compounds), uniqueN(cdt$id)))
cat(sprintf("%d compound pairs with both a fingerprint similarity and a measured target profile\n", length(tv)))
cat(sprintf("Spearman(Tanimoto, target-profile Jaccard) = %.3f\n", cor(tv, jv, method = "spearman")))
cat(sprintf("mean Tanimoto for pairs sharing >= 50 %% of their targets: %.3f | sharing < 10 %%: %.3f\n",
            mean(tv[jv >= 0.5]), mean(tv[jv < 0.1])))
cat(sprintf("pairs with Tanimoto >= 0.6: %d | their mean target-profile Jaccard %.3f (all pairs %.3f)\n",
            sum(tv >= 0.6), mean(jv[tv >= 0.6]), mean(jv)))


In [ ]:
set.seed(0); samp <- sample(length(tv), 20000)
pc1 <- ggplot(data.table(t = tv[samp], j = jv[samp]), aes(t, j)) +
  geom_point(size = 0.35, alpha = 0.1, colour = PAL[["blue"]]) +
  labs(title = "Structure vs target sharing (rho = 0.28)",
       x = "Tanimoto similarity (Morgan, 2048 bits)", y = "target-profile Jaccard")
bins <- list(c(0, 0.2), c(0.2, 0.3), c(0.3, 0.4), c(0.4, 0.6), c(0.6, 1.01))
bdt <- rbindlist(lapply(bins, function(b) {
  m <- tv >= b[1] & tv < b[2]
  data.table(mid = mean(b), n = sum(m), mean_j = mean(jv[m]), se = sd(jv[m]) / sqrt(sum(m)))
}))
pc2 <- ggplot(bdt, aes(mid, mean_j)) + geom_line(colour = PAL[["orange"]]) +
  geom_errorbar(aes(ymin = mean_j - se, ymax = mean_j + se), width = 0.02, colour = PAL[["orange"]]) +
  geom_point(colour = PAL[["orange"]], size = 1.6) +
  geom_hline(yintercept = mean(jv), linetype = "dotted") +
  annotate("text", x = 0.1, y = mean(jv) + 0.03, label = "mean over all pairs", size = 2.4, hjust = 0) +
  labs(title = "The implication runs one way",
       x = "Tanimoto similarity bin", y = "mean target-profile Jaccard")
print(pc1 | pc2)
knitr::kable(bdt, digits = 3, caption = "Target-profile overlap by similarity bin")



<p class="caption"><b>Figure 7.</b> Left: a sample of 20,000 of the 180,901 compound pairs, fingerprint similarity
against the Jaccard overlap of their measured target sets; the rank correlation over all pairs is 0.28. Right: the
same data binned — pairs below Tanimoto 0.2 share 12 % of their targets on average, pairs above 0.6 share 83 %, with
a monotone rise in between and standard errors too small to see.</p>

The relation is strong in one direction and weak in the other, which is exactly what similarity-based target
inference assumes and what a careless reading of a correlation of 0.28 would miss. **Chemically similar compounds do
have similar target profiles** — above Tanimoto 0.6 the mean overlap is 0.826 against a 0.163 baseline, a five-fold
enrichment — so predicting targets for a close analogue of a profiled compound is sound. **Dissimilar compounds often
share targets anyway**, because a kinase hinge binder can be reached from many chemotypes, so a low similarity is not
evidence of a different profile; the mean Tanimoto of pairs sharing at least half their targets is only 0.228. In
network terms: chemical similarity is a sufficient but not a necessary condition for an edge to be shared, and a
similarity network is therefore a conservative, high-precision proxy for the measured one.

## 9. Drawing a network, and when not to

A network figure is worth its space when the reader can extract something from the *positions* and the *shapes*.
Spring layouts of a thousand nodes — the hairball — almost never qualify: the layout is a stochastic compromise, a
change of seed moves everything, and no quantity can be read off reliably. Three cases justify a drawing: a small
graph; one neighbourhood shown at a readable scale and coloured by an attribute; and a schematic of a mechanism.
Anything else is better as a table or a distribution, and for interactive exploration
Cytoscape<sup class="c"><a href="#r21">[21]</a></sup> is the standard export target (`igraph::write_graph(...,
format = "graphml")`). The test before drawing is to ask which sentence of the results section the figure supports: if
that sentence is a number — a count of communities, a modularity, an enrichment — a table states it better; if it is a
shape, the drawing is the evidence and must be large enough to read.



In [ ]:
V(core)$target_name     <- unname(tname[V(core)$name])
V(core)$backbone_degree <- degree(core)
V(core)$community       <- as.integer(membership(cm)[V(core)$name])
f <- file.path(tempdir(), "backbone.graphml")      # a temporary file, not written into the unit folder
write_graph(core, f, format = "graphml")
gl <- readLines(f, warn = FALSE)
cat(sprintf("GraphML export of the backbone: %.0f kB, %d node elements, %d edge elements\n",
            file.size(f) / 1024, sum(grepl("<node ", gl)), sum(grepl("<edge ", gl))))
cat("attribute keys written:", paste(regmatches(gl, regexpr('attr\\.name="[^"]+"', gl))[1:6], collapse = ", "), "\n")
cat("\nwrite_graph(core, 'backbone.graphml', format = 'graphml') writes exactly this file for Cytoscape.\n")


In [ ]:
set.seed(0)
par(mfrow = c(1, 2), mar = c(0.5, 0.5, 2.2, 0.5))
lay <- layout_with_fr(g, niter = 300)
plot(g, layout = lay, vertex.size = ifelse(V(g)$type, 2.2, 1.4),
     vertex.color = ifelse(V(g)$type, PAL[["orange"]], PAL[["blue"]]),
     vertex.frame.color = NA, vertex.label = NA, edge.width = 0.2, edge.color = "#00000012",
     main = "1,081 nodes, 16,047 edges: a hairball")
top_t <- names(sort(tdeg, decreasing = TRUE))[1:12]
sel_c <- head(compounds[sapply(compounds, function(c_) sum(Ad[match(c_, compounds), match(top_t, targets)]) >= 6)], 60)
S <- induced_subgraph(g, c(top_t, sel_c))
pos <- rbind(cbind(0, seq_along(top_t)),
             cbind(1, seq_along(sel_c) * length(top_t) / max(1, length(sel_c))))
rownames(pos) <- c(top_t, sel_c); pos <- pos[V(S)$name, ]
plot(S, layout = pos, rescale = FALSE, xlim = c(-1.05, 1.03), ylim = c(0.5, 12.5), asp = 0,
     vertex.size = ifelse(V(S)$name %in% top_t, 7, 2.5),
     vertex.color = ifelse(V(S)$name %in% top_t, PAL[["orange"]], PAL[["blue"]]),
     vertex.frame.color = NA,
     vertex.label = ifelse(V(S)$name %in% top_t, tname[V(S)$name], NA),
     vertex.label.cex = 0.38, vertex.label.dist = 3.4, vertex.label.degree = pi,
     vertex.label.color = "black", edge.width = 0.3, edge.color = "#00000040",
     main = sprintf("12 busiest targets, %d profiled compounds", length(sel_c)))



<p class="caption"><b>Figure 8.</b> The same data drawn twice. Left: all 1,081 nodes in a Fruchterman-Reingold
layout; dense, unlabelled and dominated by layout artefacts. Right: the twelve busiest targets (orange, labelled) with
the compounds profiled against at least six of them (blue), drawn as a bipartite layout — here a reader can see that
almost every broadly profiled compound touches both ROCK kinases and most of the tyrosine kinases, which is the
sentence this section needs. The left panel belongs in no paper; the right panel is a figure.</p>

<div class="exercise"><h4>Exercise 3</h4>
<p>Open the exported <code>backbone.graphml</code> in Cytoscape, colour the nodes by the <code>community</code>
attribute and size them by <code>backbone_degree</code>. Then write the two-sentence caption you would use if the
figure had to show all 437 backbone targets instead of twelve — and explain why the table of community sizes and
kinase fractions from section 4 serves the reader better.</p></div>

## 10. Summary

This unit analysed a measured drug-target network: 602 compounds, 479 human single-protein targets, 16,047 activity
edges, one connected component, density 0.0275. Ninety-three per cent of the compounds bind more than one target —
real polypharmacology, with a median compound degree of 5 and a maximum of 386, and a median target degree of 26
against ROCK1's 344. Before any interpretation, two selection biases were measured rather than asserted: 79 % of the
targets are named "kinase" and carry 83 % of the edges, so the graph is a ROCK-kinase neighbourhood and not a sample
of pharmacological space; and the rank correlation between a compound's degree and the number of measurements behind
its edges is 0.93, so degree records assay effort at least as much as promiscuity. ChEMBL's protein-class annotation
was empty for all 479 targets, an annotation gap worked around with target names rather than hidden.

The target projection has 100,146 edges and density 0.875 untreated — no usable structure — and was reduced to a
backbone of 12,810 edges at Jaccard 0.3, leaving 42 targets isolated. Greedy modularity found 17 communities at
Q = 0.304 (Louvain, 13 at Q = 0.325), the two largest holding 179 and 177 targets of which 91 % and 79 % are named
"kinase", plus a small non-kinase community dominated by carbonic anhydrases and a four-member cytochrome P450
community — the counter-screens and ADME panels of a kinase programme. A bipartite degree-preserving null, written
explicitly so that swaps exchange target endpoints, showed why both numbers need a null: the mean Jaccard over target
pairs is 0.1482 observed against 0.1424 +/- 0.0003 randomised (z = 17, a 4 % effect), while the backbone holds 12,810
edges against 2,079 +/- 122 (z = 88, a six-fold excess). Structure lives in the tail, not the average. Centrality told
the same story about provenance: exact betweenness is led by ROCK1 (0.314) and ROCK2 (0.096) and then by the
most-assayed compounds, and correlates with degree at Spearman 0.86.

The selectivity question was answered directly from the 276 compounds measured at both ROCK isoforms: potencies
correlate at r = 0.870 (Spearman 0.851), the mean difference is -0.50 log units in favour of ROCK2, 15.6 % of
compounds exceed tenfold selectivity and the extreme is -3.55; 68 compounds are measured at ROCK1 only and 34 at
ROCK2 only, so a missing edge is a missing experiment and not an inactivity. The panel-wide version of the same
question — the gap between the best and tenth-best potency of the 174 compounds profiled at twenty targets or more —
has a median of 1.47 log units and falls with the number of targets measured (Spearman -0.379). Link prediction gave
the sharpest methodological result: three-step path counting reaches ROC-AUC 0.927 against uniformly drawn negatives —
but so does preferential attachment at 0.921, which is the warning — and against negatives matched on target degree
the three baselines fall to 0.561, 0.586 and 0.506. Finally, comparing the two kinds of edge showed that chemical
similarity implies shared targets (pairs above Tanimoto 0.6 share 83 % of their targets against a 16 % baseline) while
the converse fails (overall Spearman 0.28), so a similarity network is a high-precision, low-recall proxy for a
measured one.

The three habits to keep: state how every edge was defined and with which threshold; compare every network statistic
with a degree-preserving null before interpreting it; and choose the negatives of a graph benchmark as carefully as
the decoys of a screening benchmark. A real project adds the graphs this unit did not
download — STRING<sup class="c"><a href="#r7">[7]</a></sup> for protein interactions, Open
Targets<sup class="c"><a href="#r20">[20]</a></sup> for target-disease evidence,
Hetionet<sup class="c"><a href="#r6">[6]</a></sup> for a knowledge graph — and applies the same
machinery<sup class="c"><a href="#r22">[22]</a></sup><sup class="c"><a href="#r23">[23]</a></sup><sup class="c"><a href="#r24">[24]</a></sup>,
including the small-world and network-medicine
framings<sup class="c"><a href="#r11">[11]</a></sup><sup class="c"><a href="#r25">[25]</a></sup>.

<div class="box bridge"><h4>Bridge</h4>
<p>Within this course: the fingerprints and similarity come from <a href="../U13_Features_Dimensionality/index.html">Unit 13</a>,
the clustering from <a href="../U15_Clustering/index.html">Unit 15</a>, the hold-out protocol from
<a href="../U12_Validation/index.html">Unit 12</a>, the permutation tests from
<a href="../U8_Hypothesis_Testing/index.html">Unit 8</a>, and the negative-set argument from
<a href="../U19_Virtual_Screening/index.html">Unit 19</a>, whose sparse Tanimoto algebra is reused here; the omics
counterpart of a neighbourhood test is in <a href="../U21_Omics/index.html">Unit 21</a>, and the replica-level
statistics of <a href="../U20_MD_Analytics/index.html">Unit 20</a> are the same discipline applied to simulation
output. The Python edition of this unit, built with NetworkX and RDKit, is
<a href="../../Python/U22_Networks/index.html">here</a>. In the sister courses: the targets and their structures are
introduced in <a href="../../../BGM324/02_Protein_Structure_Prediction/index.html">BGM324 unit 2</a>
and the structure-based design context in
<a href="../../../BGM324/04_Computer_Aided_Drug_Design/index.html">BGM324 unit 4</a>;
chemical similarity and chemical space are developed in
<a href="../../../Cheminformatics/U8_Chemical_Space/index.html">Cheminformatics unit 8</a> and
<a href="../../../Cheminformatics/U11_Similarity_SAR/index.html">unit 11</a>; the evaluation protocol behind
the link-prediction section is in <a href="../../../BGM411/R/index.html">BGM411 (R edition)</a>.</p></div>

## 11. Self-assessment

<details><summary>1. Your bipartite projection has density 0.875. What went wrong, and what do you do?</summary>
<p>Nothing went wrong - projections of dense bipartite graphs are nearly complete, because almost every pair of
targets shares at least one compound. Threshold the projection into a backbone (here Jaccard 0.3, which keeps 12,810
of 100,146 edges), report the threshold, and check how many nodes it isolates (42 here).</p></details>

<details><summary>2. Why is the Jaccard coefficient preferable to the raw count of shared compounds as a projection weight?</summary>
<p>Because the raw count grows with the degrees: ROCK1 shares many compounds with everything simply because 344
compounds were measured against it. The Jaccard coefficient divides by the union, so a pair of small, highly
overlapping target sets can outrank a pair of large, weakly overlapping ones.</p></details>

<details><summary>3. The mean Jaccard is 4 % above its degree-preserving null but the backbone has six times more edges than the null. Which do you report?</summary>
<p>Both, and the explanation: the average is almost fixed by the degree sequence, while the excess is concentrated in
the upper tail. A statistic with a tail (number of strong links, modularity, a specific pair's overlap) is the one
worth testing; an average that a null reproduces to within 4 % carries almost no information.</p></details>

<details><summary>4. Why must a degree-preserving swap in a bipartite graph exchange endpoints on the same side?</summary>
<p>Because swapping across sides would create compound-compound or target-target edges and destroy the bipartite
structure, changing the object rather than randomising it. The swap used here takes two compound-target edges and
exchanges their targets, which preserves every degree on both sides.</p></details>

<details><summary>5. A compound has no edge to ROCK2. Is it ROCK1-selective?</summary>
<p>No conclusion is possible. The absence of an edge means no measurement with a pChEMBL value exists in this
extraction - 68 compounds here are measured at ROCK1 and not at ROCK2. Selectivity requires both measurements; the 276
compounds that have them are the only ones about which the question can be asked.</p></details>

<details><summary>6. Your link-prediction model reaches ROC-AUC 0.93. What is the first thing to check?</summary>
<p>How the negatives were drawn, and what a degree-only baseline scores. Here preferential attachment - the product of
the two degrees, which knows nothing about structure - reached 0.921 against uniform negatives, and against
degree-matched negatives every method collapsed to 0.51-0.59. A benchmark is as hard as its negatives.</p></details>

<details><summary>7. In a bipartite graph the usual common-neighbour score is always zero. Why, and what replaces it?</summary>
<p>A compound and a target are on opposite sides, so they have no neighbours in common; the shortest informative path
has length three. The bipartite analogues count three-step paths (compound - target - compound - target), or weight
them by the similarity of the two compounds' target profiles.</p></details>

<details><summary>8. Degree correlates with the number of measurements at Spearman 0.93. What does that do to a "most promiscuous compounds" list?</summary>
<p>It makes it mostly a list of the most thoroughly profiled compounds. To rank promiscuity one must condition on
assay effort - for instance by restricting to compounds screened in a common panel, or by modelling the hit rate
(targets hit divided by targets tested) rather than the raw degree. The panel-wide selectivity index of section 6 is
one such conditioning, and it still correlates with effort at -0.38.</p></details>

<details><summary>9. Your communities are 91 % kinases. Is that biology?</summary>
<p>It is provenance. The compounds were selected as ROCK1 ligands and were therefore assayed mainly in kinase panels,
so the graph can hardly contain anything else; 79 % of all targets in it are named "kinase". The community structure
faithfully describes the data set and says nothing about pharmacological space in general.</p></details>

<details><summary>10. Greedy modularity returns 17 communities at Q = 0.304 and Louvain returns 13 at Q = 0.325 on the same backbone. Which is correct?</summary>
<p>Neither is "the" answer: both are heuristics for the same NP-hard objective, and a higher Q simply means a better
optimum was found. Report the algorithm, its parameters and the modularity, and check that the conclusion you draw
(here, that the communities are kinase panels) survives both partitions. The resolution limit means that neither can
reliably detect families smaller than a few nodes.</p></details>

<details><summary>11. ChEMBL returned no protein class for any target. How should the analysis report that?</summary>
<p>Explicitly: state the coverage (0 of 479 here), use a declared surrogate - a text match on the target name in this
unit - and label every class-based statement as derived from that surrogate. Silently dropping the annotation, or
quietly substituting another source without saying so, is what makes an analysis unreproducible.</p></details>

<details><summary>12. What minimum information must accompany any network result?</summary>
<p>The node and edge definitions, every threshold used, the number of nodes and edges, the provenance and selection
rule of the data, the software and version, the null model with the number of randomisations, and - for link
prediction - how the negatives were sampled.</p></details>

## Glossary

<dl class="gloss">
<dt>Bipartite graph</dt><dd>A graph whose nodes split into two sets with edges only between the sets, such as compounds and targets; marked in igraph by the logical vertex attribute <code>type</code>.</dd>
<dt>Projection</dt><dd>Graph on one side of a bipartite network, joining two nodes that share a neighbour; weighted by shared counts or by a Jaccard coefficient.</dd>
<dt>Backbone</dt><dd>The subgraph left after discarding projection edges below a weight threshold; a projection is uninterpretable without one.</dd>
<dt>Degree</dt><dd>Number of edges at a node; here the number of targets a compound was measured at, or compounds a target was screened with.</dd>
<dt>pChEMBL</dt><dd>-log10 of the molar concentration of a measured activity (IC50, Ki, Kd); 7 means 100 nM, and one unit is a factor of ten.</dd>
<dt>Polypharmacology / selectivity</dt><dd>Binding several targets / preferring one target over its relatives; a tenfold preference is one pChEMBL unit.</dd>
<dt>Betweenness centrality</dt><dd>Fraction of shortest paths passing through a node; in a designed data set it reproduces the design before it reveals biology.</dd>
<dt>Eigenvector centrality</dt><dd>Importance defined recursively by the importance of neighbours; well defined only on a connected graph, and scaled by igraph so that the maximum is 1.</dd>
<dt>Bipartite clustering coefficient</dt><dd>Mean neighbourhood overlap (here the Jaccard "dot" mode) between a node and the nodes two hops away; the bipartite replacement for a triangle count.</dd>
<dt>Modularity Q</dt><dd>Fraction of edges inside communities minus the fraction expected in a degree-preserving random graph.</dd>
<dt>Resolution limit</dt><dd>Inability of modularity maximisation to detect communities below a size that grows with the graph.</dd>
<dt>Degree-preserving randomisation</dt><dd>Rewiring that keeps every node's degree and destroys everything else; in a bipartite graph the swap must exchange endpoints on the same side.</dd>
<dt>Link prediction</dt><dd>Scoring absent edges by plausibility; bipartite baselines count three-step paths, weight them by profile similarity, or multiply degrees.</dd>
<dt>Hard negatives</dt><dd>Non-edges chosen to be plausible - here matched on target degree - rather than uniformly at random.</dd>
<dt>Assay-effort bias</dt><dd>Degree reflecting how much a node has been measured rather than its biology; here Spearman 0.93 between compound degree and number of measurements.</dd>
<dt>Annotation gap</dt><dd>A metadata field that is absent for part or all of the data (here protein class for 479 of 479 targets), to be reported and worked around explicitly.</dd>
</dl>

## References and further reading

<ol class="refs">
<li id="r1">Barabasi AL, Gulbahce N, Loscalzo J. Network medicine: a network-based approach to human disease. <em>Nat Rev Genet</em>. 2011;12(1):56-68. <a href="https://doi.org/10.1038/nrg2918">doi:10.1038/nrg2918</a></li>
<li id="r2">Hopkins AL. Network pharmacology: the next paradigm in drug discovery. <em>Nat Chem Biol</em>. 2008;4(11):682-690. <a href="https://doi.org/10.1038/nchembio.118">doi:10.1038/nchembio.118</a></li>
<li id="r3">Yildirim MA, Goh KI, Cusick ME, Barabasi AL, Vidal M. Drug-target network. <em>Nat Biotechnol</em>. 2007;25(10):1119-1126. <a href="https://doi.org/10.1038/nbt1338">doi:10.1038/nbt1338</a></li>
<li id="r4">Blondel VD, Guillaume JL, Lambiotte R, Lefebvre E. Fast unfolding of communities in large networks. <em>J Stat Mech</em>. 2008;2008(10):P10008. <a href="https://doi.org/10.1088/1742-5468/2008/10/P10008">doi:10.1088/1742-5468/2008/10/P10008</a></li>
<li id="r5">Traag VA, Waltman L, van Eck NJ. From Louvain to Leiden: guaranteeing well-connected communities. <em>Sci Rep</em>. 2019;9:5233. <a href="https://doi.org/10.1038/s41598-019-41695-z">doi:10.1038/s41598-019-41695-z</a></li>
<li id="r6">Himmelstein DS, Lizee A, Hessler C, et al. Systematic integration of biomedical knowledge prioritizes drugs for repurposing. <em>eLife</em>. 2017;6:e26726. <a href="https://doi.org/10.7554/eLife.26726">doi:10.7554/eLife.26726</a></li>
<li id="r7">Szklarczyk D, Kirsch R, Koutrouli M, et al. The STRING database in 2023: protein-protein association networks and functional enrichment analyses. <em>Nucleic Acids Res</em>. 2023;51(D1):D638-D646. <a href="https://doi.org/10.1093/nar/gkac1000">doi:10.1093/nar/gkac1000</a></li>
<li id="r8">Keiser MJ, Roth BL, Armbruster BN, Ernsberger P, Irwin JJ, Shoichet BK. Relating protein pharmacology by ligand chemistry. <em>Nat Biotechnol</em>. 2007;25(2):197-206. <a href="https://doi.org/10.1038/nbt1284">doi:10.1038/nbt1284</a></li>
<li id="r9">Keiser MJ, Setola V, Irwin JJ, et al. Predicting new molecular targets for known drugs. <em>Nature</em>. 2009;462(7270):175-181. <a href="https://doi.org/10.1038/nature08506">doi:10.1038/nature08506</a></li>
<li id="r10">Paolini GV, Shapland RHB, van Hoorn WP, Mason JS, Hopkins AL. Global mapping of pharmacological space. <em>Nat Biotechnol</em>. 2006;24(7):805-815. <a href="https://doi.org/10.1038/nbt1228">doi:10.1038/nbt1228</a></li>
<li id="r11">Newman MEJ. The structure and function of complex networks. <em>SIAM Rev</em>. 2003;45(2):167-256. <a href="https://doi.org/10.1137/S003614450342480">doi:10.1137/S003614450342480</a></li>
<li id="r12">Newman MEJ, Girvan M. Finding and evaluating community structure in networks. <em>Phys Rev E</em>. 2004;69(2):026113. <a href="https://doi.org/10.1103/PhysRevE.69.026113">doi:10.1103/PhysRevE.69.026113</a></li>
<li id="r13">Fortunato S, Barthelemy M. Resolution limit in community detection. <em>Proc Natl Acad Sci USA</em>. 2007;104(1):36-41. <a href="https://doi.org/10.1073/pnas.0605965104">doi:10.1073/pnas.0605965104</a></li>
<li id="r14">Clauset A, Newman MEJ, Moore C. Finding community structure in very large networks. <em>Phys Rev E</em>. 2004;70(6):066111. <a href="https://doi.org/10.1103/PhysRevE.70.066111">doi:10.1103/PhysRevE.70.066111</a></li>
<li id="r15">Barabasi AL, Albert R. Emergence of scaling in random networks. <em>Science</em>. 1999;286(5439):509-512. <a href="https://doi.org/10.1126/science.286.5439.509">doi:10.1126/science.286.5439.509</a></li>
<li id="r16">Maslov S, Sneppen K. Specificity and stability in topology of protein networks. <em>Science</em>. 2002;296(5569):910-913. <a href="https://doi.org/10.1126/science.1065103">doi:10.1126/science.1065103</a></li>
<li id="r17">Adamic LA, Adar E. Friends and neighbors on the Web. <em>Soc Networks</em>. 2003;25(3):211-230. <a href="https://doi.org/10.1016/S0378-8733(03)00009-1">doi:10.1016/S0378-8733(03)00009-1</a></li>
<li id="r18">Liben-Nowell D, Kleinberg J. The link-prediction problem for social networks. <em>J Am Soc Inf Sci Technol</em>. 2007;58(7):1019-1031. <a href="https://doi.org/10.1002/asi.20591">doi:10.1002/asi.20591</a></li>
<li id="r19">Lu L, Zhou T. Link prediction in complex networks: a survey. <em>Physica A</em>. 2011;390(6):1150-1170. <a href="https://doi.org/10.1016/j.physa.2010.11.027">doi:10.1016/j.physa.2010.11.027</a></li>
<li id="r20">Ochoa D, Hercules A, Carmona M, et al. The next-generation Open Targets Platform: reimagined, redesigned, rebuilt. <em>Nucleic Acids Res</em>. 2023;51(D1):D1353-D1359. <a href="https://doi.org/10.1093/nar/gkac1046">doi:10.1093/nar/gkac1046</a></li>
<li id="r21">Shannon P, Markiel A, Ozier O, et al. Cytoscape: a software environment for integrated models of biomolecular interaction networks. <em>Genome Res</em>. 2003;13(11):2498-2504. <a href="https://doi.org/10.1101/gr.1239303">doi:10.1101/gr.1239303</a></li>
<li id="r22">Guney E, Menche J, Vidal M, Barabasi AL. Network-based in silico drug efficacy screening. <em>Nat Commun</em>. 2016;7:10331. <a href="https://doi.org/10.1038/ncomms10331">doi:10.1038/ncomms10331</a></li>
<li id="r23">Menche J, Sharma A, Kitsak M, et al. Uncovering disease-disease relationships through the incomplete interactome. <em>Science</em>. 2015;347(6224):1257601. <a href="https://doi.org/10.1126/science.1257601">doi:10.1126/science.1257601</a></li>
<li id="r24">Cheng F, Kovacs IA, Barabasi AL. Network-based prediction of drug combinations. <em>Nat Commun</em>. 2019;10:1197. <a href="https://doi.org/10.1038/s41467-019-09186-x">doi:10.1038/s41467-019-09186-x</a></li>
<li id="r25">Watts DJ, Strogatz SH. Collective dynamics of 'small-world' networks. <em>Nature</em>. 1998;393(6684):440-442. <a href="https://doi.org/10.1038/30918">doi:10.1038/30918</a></li>
<li id="r26">Zhou T, Lu L, Zhang YC. Predicting missing links via local information. <em>Eur Phys J B</em>. 2009;71(4):623-630. <a href="https://doi.org/10.1140/epjb/e2009-00335-8">doi:10.1140/epjb/e2009-00335-8</a></li>
<li id="r27">Bemis GW, Murcko MA. The properties of known drugs. 1. Molecular frameworks. <em>J Med Chem</em>. 1996;39(15):2887-2893. <a href="https://doi.org/10.1021/jm9602928">doi:10.1021/jm9602928</a></li>
<li id="r28">Zdrazil B, Felix E, Hunter F, et al. The ChEMBL Database in 2023: a drug discovery platform spanning multiple bioactivity data types and time periods. <em>Nucleic Acids Res</em>. 2024;52(D1):D1180-D1192. <a href="https://doi.org/10.1093/nar/gkad1004">doi:10.1093/nar/gkad1004</a></li>
<li id="r29">Rogers D, Hahn M. Extended-connectivity fingerprints. <em>J Chem Inf Model</em>. 2010;50(5):742-754. <a href="https://doi.org/10.1021/ci100050t">doi:10.1021/ci100050t</a></li>
<li id="r30">Ihaka R, Gentleman R. R: a language for data analysis and graphics. <em>J Comput Graph Stat</em>. 1996;5(3):299-314. <a href="https://doi.org/10.1080/10618600.1996.10474713">doi:10.1080/10618600.1996.10474713</a></li>
<li id="r31">Wickham H. <em>ggplot2: Elegant Graphics for Data Analysis</em>. 2nd ed. Springer; 2016. <a href="https://doi.org/10.1007/978-3-319-24277-4">doi:10.1007/978-3-319-24277-4</a></li>
<li id="r32">Serrano MA, Boguna M, Vespignani A. Extracting the multiscale backbone of complex weighted networks. <em>Proc Natl Acad Sci USA</em>. 2009;106(16):6483-6488. <a href="https://doi.org/10.1073/pnas.0808904106">doi:10.1073/pnas.0808904106</a></li>
<li id="r33">Latapy M, Magnien C, Del Vecchio N. Basic notions for the analysis of large two-mode networks. <em>Soc Networks</em>. 2008;30(1):31-48. <a href="https://doi.org/10.1016/j.socnet.2007.04.006">doi:10.1016/j.socnet.2007.04.006</a></li>
</ol>

<h4>Textbooks</h4>
<ul>
<li>Barabasi AL. <em>Network Science</em>. Cambridge University Press; 2016. - Free online; degree distributions, random-graph nulls, communities and robustness.</li>
<li>Newman MEJ. <em>Networks</em>. 2nd ed. Oxford University Press; 2018. - The reference text for the descriptors used here.</li>
<li>Kolaczyk ED, Csardi G. <em>Statistical Analysis of Network Data with R</em>. 2nd ed. Springer; 2020. - The igraph-based companion to this unit, by one of igraph's authors.</li>
<li>Leach AR, Gillet VJ. <em>An Introduction to Chemoinformatics</em>. Rev. ed. Springer; 2007. - Similarity networks and chemical space.</li>
</ul>

<h4>Online resources</h4>
<ul>
<li><code>igraph</code> R manual, including the bipartite and community-detection functions used here: <a href="https://r.igraph.org/">r.igraph.org</a>.</li>
<li>Cytoscape, for interactive inspection and publication-quality layouts: <a href="https://cytoscape.org/">cytoscape.org</a>.</li>
<li>ChEMBL, the source of the frozen drug-target table: <a href="https://www.ebi.ac.uk/chembl/">ebi.ac.uk/chembl</a>.</li>
<li>STRING, Open Targets and Hetionet, where a real project obtains interaction, target-disease and knowledge graphs: <a href="https://string-db.org/">string-db.org</a>, <a href="https://platform.opentargets.org/">platform.opentargets.org</a>, <a href="https://het.io/">het.io</a>.</li>
<li>CRAN Task View "Graphical Models and Network Analysis": <a href="https://cran.r-project.org/view=GraphicalModels">cran.r-project.org/view=GraphicalModels</a>.</li>
</ul>

